### IBM style decoding 

In [1]:
import pickle
from typing import Any, Optional



In [2]:
import pickle
from nltk.translate import IBMModel1, IBMModel2
from nltk.translate import AlignedSent


### read learner dataset; 

## https://raw.githubusercontent.com/HillZhang1999/MuCGEC/refs/heads/main/data/MuCGEC/MuCGEC_dev.txt
file_name = "MuCGEC_dev.txt" 
with open (file_name, "r") as file_in:
    data = file_in.read().splitlines()

sentence_pairs = []

for line in data:
    line = line.split('\t')
    source = list(line[1])
    ref_list = line[2:]
    for ref in ref_list:
        target = list(ref)
        sentence_pairs.append((source, target))

bitext = []

for i in sentence_pairs:
    bitext.append(AlignedSent(i[0], i[1]))

In [3]:
ibm_alignment2 = IBMModel2(bitext, 5)

In [ ]:
# import pickle

# def export_ibm_tables(ibm_model):
#     """
#     Turn NLTK IBMModel(1/2) into a pickle-safe dict:
#       - 'type': 'ibm1' or 'ibm2'
#       - 't': {e: {f: prob}}
#       - 'a': {(i,j,l_e,l_f): prob}   # only for IBM2
#     """
#     # translation table: dict[e][f] -> float
#     t_src = getattr(ibm_model, "translation_table", None)
#     if t_src is None:
#         raise TypeError("ibm_model has no translation_table")

#     t = {}
#     for e, row in t_src.items():
#         # row may be a ProbDist or defaultdict-like; normalize to plain dict
#         try:
#             # ProbDist-like
#             items = row.items()
#         except Exception:
#             # NLTK’s prob dists sometimes need .samples() / prob()
#             # fall back to iteration via keys if needed
#             items = []
#             if hasattr(row, "samples") and hasattr(row, "prob"):
#                 items = [(f, row.prob(f)) for f in row.samples()]
#             else:
#                 items = row.items()  # best effort
#         t[e] = {f: float(p) for f, p in items}

#     payload = {"type": "ibm1", "t": t}

#     # alignment table (only for IBM2)
#     a_src = getattr(ibm_model, "alignment_table", None)
#     if a_src is not None:
#         try:
#             a_items = a_src.items()
#         except Exception:
#             # AlignmentTable sometimes needs iteration over keys
#             a_items = [(k, a_src[k]) for k in a_src]
#         a = {tuple(k): float(v) for k, v in a_items}
#         payload["type"] = "ibm2"
#         payload["a"] = a

#     return payload

# def save_ibm_as_tables(ibm_model, path):
#     data = export_ibm_tables(ibm_model)
#     with open(path, "wb") as f:
#         pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

# def load_ibm_tables(path):
#     with open(path, "rb") as f:
#         data = pickle.load(f)
#     return data   # feed this to your decoder

# save_ibm_as_tables(ibm_alignment1, "model_learner_ibm1_tables.pkl")

In [4]:
import pickle
from collections import abc
from typing import Any, Dict, Iterable, Tuple

# ---------- Helpers ----------

def _as_plain_dict(d) -> Dict:
    """Best-effort convert an NLTK probdist/defaultdict-like object to a plain dict."""
    if d is None:
        return {}
    # If it already behaves like a dict
    if hasattr(d, "items"):
        try:
            return {k: float(v) for k, v in d.items()}
        except Exception:
            pass
    # NLTK probdist: samples()/prob()
    if hasattr(d, "samples") and hasattr(d, "prob"):
        return {k: float(d.prob(k)) for k in d.samples()}
    # Try mapping interface
    try:
        return {k: float(d[k]) for k in d}
    except Exception:
        # Last resort: iterate attributes (unlikely)
        return dict(d)

def _flatten_alignment_table_nested(a_src: Any) -> Dict[Tuple[int,int,int,int], float]:
    """
    Flatten nested IBM2 alignment table a[i][j][l_e][l_f] -> prob
    into {(i,j,l_e,l_f): prob}.
    Accepts: nested dicts or an already-flat dict.
    """
    # Already flat?
    if hasattr(a_src, "items"):
        # Peek at a key
        try:
            for k, v in a_src.items():
                if isinstance(k, tuple) and len(k) == 4:
                    # Flat
                    return {(int(k[0]), int(k[1]), int(k[2]), int(k[3])): float(v) for k, v in a_src.items()}
                break
        except Exception:
            pass

    # Treat as nested mappings: a[i][j][l_e][l_f]
    flat = {}
    try:
        for i, a_i in a_src.items():
            # a_i ~ dict[j] -> dict[l_e] -> dict[l_f] -> prob
            if not hasattr(a_i, "items"):
                continue
            for j, a_ij in a_i.items():
                if not hasattr(a_ij, "items"):
                    continue
                for l_e, a_ijl in a_ij.items():
                    # a_ijl might be probdist or dict
                    inner = _as_plain_dict(a_ijl)
                    for l_f, p in inner.items():
                        flat[(int(i), int(j), int(l_e), int(l_f))] = float(p)
        return flat
    except Exception:
        # Some AlignmentTable supports direct iteration over keys
        try:
            return {tuple(k): float(a_src[k]) for k in a_src}
        except Exception as e:
            raise TypeError(f"Unrecognized alignment_table structure: {type(a_src)}") from e

# ---------- Exporter ----------

def export_ibm_tables(ibm_model: Any) -> Dict[str, Any]:
    """
    Convert NLTK IBM1/IBM2 model to a pickle-safe dict:
      {'type': 'ibm1'|'ibm2', 't': {e: {f: prob}}, 'a': {(i,j,l_e,l_f): prob}?}
    """
    # Translation table
    t_src = getattr(ibm_model, "translation_table", None)
    if t_src is None:
        raise TypeError("ibm_model has no translation_table")
    t = {}
    for e, row in t_src.items():
        row_dict = _as_plain_dict(row)
        t[e] = {f: float(p) for f, p in row_dict.items()}

    payload = {"type": "ibm1", "t": t}

    # Alignment/distortion table (IBM2)
    a_src = getattr(ibm_model, "alignment_table", None)
    if a_src is not None:
        a = _flatten_alignment_table_nested(a_src)
        payload["type"] = "ibm2"
        payload["a"] = a

    return payload

def save_ibm_as_tables(ibm_model: Any, path: str) -> None:
    data = export_ibm_tables(ibm_model)
    with open(path, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_ibm_tables(path: str) -> Dict[str, Any]:
    with open(path, "rb") as f:
        return pickle.load(f)

# ---------- Matching decoder (tables -> alignments) ----------

from typing import List, Optional, Tuple

def build_tprob_from_tables(tbl: Dict):
    ttab = tbl["t"]
    def tprob(e, f):
        d = ttab.get(e)
        if d is None and e == "NULL":
            d = ttab.get(None)
        return 0.0 if d is None else float(d.get(f, 0.0))
    return tprob

def build_aprob_from_tables(tbl: Dict):
    atab = tbl.get("a", None)
    if atab is not None:
        def aprob(i, j, l_e, l_f):
            return float(atab.get((i, j, l_e, l_f), 0.0))
        return aprob
    else:
        # IBM1 fallback: uniform over i (including NULL)
        def aprob(i, j, l_e, l_f):
            return 1.0 / float(l_e)
        return aprob

def ibm_decode_chars_tables(
    tables: Dict[str, Any],
    src_chars: List[str],
    tgt_chars: List[str],
    use_null: bool = True,
    keep_null_as_none: bool = True,
    epsilon_t: float = 1e-12,
    epsilon_a: float = 1e-12,
) -> List[Tuple[Optional[int], int]]:
    tprob = build_tprob_from_tables(tables)
    aprob = build_aprob_from_tables(tables)

    E = [None] + src_chars if use_null else list(src_chars)
    l_e, l_f = len(E), len(tgt_chars)

    pairs: List[Tuple[Optional[int], int]] = []
    for j, f_j in enumerate(tgt_chars, start=1):
        best_i, best_score = (0 if use_null else 0), -1.0
        for i, e_i in enumerate(E):
            t = tprob(e_i, f_j) if e_i is not None else (tprob(None, f_j) or tprob("NULL", f_j))
            a = aprob(i, j, l_e, l_f)
            if t == 0.0: t = epsilon_t
            if a == 0.0: a = epsilon_a
            s = t * a
            if s > best_score:
                best_score, best_i = s, i
        tgt_idx = j - 1
        if use_null and best_i == 0:
            pairs.append((None if keep_null_as_none else -1, tgt_idx))
        else:
            src_idx = best_i - (1 if use_null else 0)
            pairs.append((src_idx, tgt_idx))
    return pairs

# ---------- Example usage ----------
save_ibm_as_tables(ibm_alignment2, "model_dialect_ibm2_tables.pkl")
tbl2 = load_ibm_tables("model_dialect_ibm2_tables.pkl")
# pairs = ibm_decode_chars_tables(tbl2, list("因为在冰箱里没什么东西也做很好吃的菜。"),
#                                       list("即使在冰箱里没什么东西也能做很好吃的菜。"))
# print(pairs[:10])

In [6]:
from typing import Any, Dict, List, Optional, Tuple

def build_tprob_from_tables(tbl: Dict) :
    """
    t: dict[e][f] -> prob
    Handles NULL as both None and "NULL".
    """
    ttab = tbl["t"]
    def tprob(e, f):
        d = ttab.get(e)
        if d is None and e == "NULL":
            d = ttab.get(None)
        return 0.0 if d is None else float(d.get(f, 0.0))
    return tprob

def build_aprob_from_tables(tbl: Dict):
    """
    a: dict[(i,j,l_e,l_f)] -> prob   (only for IBM2)
    If missing (IBM1), return a uniform distortion model.
    """
    atab = tbl.get("a", None)

    if atab is not None:
        def aprob(i, j, l_e, l_f):
            return float(atab.get((i, j, l_e, l_f), 0.0))
        return aprob
    else:
        # IBM1: no alignment/distortion model; use uniform over i
        def aprob(i, j, l_e, l_f):
            return 1.0 / float(l_e)  # l_e includes NULL if used
        return aprob

def ibm_decode_chars_tables(
    tables: Dict[str, Any],
    src_chars: List[str],
    tgt_chars: List[str],
    use_null: bool = True,
    keep_null_as_none: bool = True,
    epsilon_t: float = 1e-12,
    epsilon_a: float = 1e-12,
) -> List[Tuple[Optional[int], int]]:
    """
    Decode IBM1/IBM2 given a dict of tables: {'type': 'ibm1'|'ibm2', 't': ..., 'a': ...?}
    Returns list of (src_index_or_None, tgt_index).
    """
    tprob = build_tprob_from_tables(tables)
    aprob = build_aprob_from_tables(tables)

    # Build source with optional NULL at position 0
    E = [None] + src_chars if use_null else list(src_chars)
    l_e, l_f = len(E), len(tgt_chars)

    pairs: List[Tuple[Optional[int], int]] = []

    for j, f_j in enumerate(tgt_chars, start=1):  # IBM uses 1..l_f
        best_i, best_score = (0 if use_null else 0), -1.0
        for i, e_i in enumerate(E):
            # lexical prob; try None then "NULL" for the null token
            t = tprob(e_i, f_j) if e_i is not None else (tprob(None, f_j) or tprob("NULL", f_j))
            a = aprob(i, j, l_e, l_f)
            if t == 0.0: t = epsilon_t
            if a == 0.0: a = epsilon_a
            s = t * a
            if s > best_score:
                best_score, best_i = s, i

        tgt_idx = j - 1
        if use_null and best_i == 0:
            pairs.append((None if keep_null_as_none else -1, tgt_idx))
        else:
            src_idx = best_i - (1 if use_null else 0)
            pairs.append((src_idx, tgt_idx))

    return pairs

# ----------------- Example -----------------
src_chars = list("因为在冰箱里没什么东西也做很好吃的菜。")
tgt_chars = list("即使在冰箱里没什么东西也能做很好吃的菜。")

tables = load_ibm_tables("model_dialect_ibm2_tables.pkl")  # your saved dict
pairs = ibm_decode_chars_tables(tables, src_chars, tgt_chars)
print(pairs)

[(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 7), (8, 8), (9, 9), (10, 10), (11, 11), (12, 12), (12, 13), (13, 14), (14, 15), (16, 16), (16, 17), (18, 18), (18, 19)]
